In [5]:
%%writefile /kaggle/working/herbinator.py
import cv2
import cv2.aruco as aruco
from ultralytics import YOLO
import serial
import time

# Charger le modèle YOLO
model = YOLO("/home/pi/herbinator/best.pt")

# Caméra
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 416)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 416)

# Arduino via UART
arduino = serial.Serial('/dev/ttyACM0', 9600, timeout=1)
time.sleep(2)
print("Arduino connecté !")

# ArUco
aruco_dict = aruco.getPredefinedDictionary(aruco.DICT_4X4_50)
parameters = aruco.DetectorParameters()
detector = aruco.ArucoDetector(aruco_dict, parameters)

def avancer():
    arduino.write(b"GO\n")

def stopper():
    arduino.write(b"STOP\n")

def demi_tour_droite():
    arduino.write(b"TURN_RIGHT_180\n")

def demi_tour_gauche():
    arduino.write(b"TURN_LEFT_180\n")

def reagir_aruco(marker_id):
    if marker_id == 0:
        print("ID 0 - Coin haut-gauche → demi-tour droite")
        stopper()
        time.sleep(0.3)
        demi_tour_droite()
        time.sleep(1)
        avancer()
    elif marker_id == 1:
        print("ID 1 - Coin haut-droit → demi-tour gauche")
        stopper()
        time.sleep(0.3)
        demi_tour_gauche()
        time.sleep(1)
        avancer()
    elif marker_id == 2:
        print("ID 2 - Coin bas-gauche → demi-tour droite")
        stopper()
        time.sleep(0.3)
        demi_tour_droite()
        time.sleep(1)
        avancer()
    elif marker_id == 3:
        print("ID 3 - Coin bas-droit → demi-tour gauche")
        stopper()
        time.sleep(0.3)
        demi_tour_gauche()
        time.sleep(1)
        avancer()

print("Herbinator démarré !")
avancer()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    corners, ids, _ = detector.detectMarkers(frame)
    if ids is not None:
        for marker_id in ids.flatten():
            reagir_aruco(int(marker_id))
        continue

    results = model(frame, imgsz=416, conf=0.5, verbose=False)
    for result in results:
        if len(result.boxes) > 0:
            classe = int(result.boxes[0].cls[0])
            confiance = float(result.boxes[0].conf[0])
            print(f"Mauvaise herbe détectée : classe {classe}, confiance {confiance:.2f}")
            stopper()
            time.sleep(3)
            avancer()

cap.release()
print("Herbinator arrêté.")

Overwriting /kaggle/working/herbinator.py
